# IBKR News API Probe

This notebook separates the two IBKR news API paths so provider permissions and BroadTape contracts are not mixed together.

- Test A: contract-specific stock news using `reqMktData(stock, "mdoff,292:BRFG+BRFUPDN+DJNL")`. This is the main path for portfolio monitoring.
- Test B: all-news BroadTape contracts using `NEWS` contracts such as `BZ:BZ_ALL`, `FLY:FLY_ALL`, and `BRF:BRF_ALL`. This needs separate API BroadTape entitlement.

`SNAPSHOT` means old headlines sent by IB when the subscription starts. `NEW` means a headline that arrived after this run started.


## 0. Settings


In [ ]:
import html
import re
import threading
import time
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

from ibapi.client import EClient
from ibapi.contract import Contract
from ibapi.wrapper import EWrapper

HOST = "127.0.0.1"
PORT = 4001
CLIENT_ID = 1901
LOCAL_TZ = ZoneInfo("Asia/Shanghai")

# Test A: stock/contract-specific news. This is the stable path for BRFG/BRFUPDN/DJNL.
CONTRACT_NEWS_PROVIDERS = "BRFG+BRFUPDN+DJNL"
CONTRACT_NEWS_SYMBOLS = [
    "AOSL", "AVAV", "IBM", "LHX", "SNOW", "VRT", "DRAM", "FCX", "IGV", "NET",
    "PUMP", "TCOM", "UUUU", "VELO", "5803", "6471", "SIVE", "IQE", "XFAB",
]

# Test B: all-news BroadTape. IB docs show these as NEWS contracts.
# If you do not have the API-specific subscription, code=200 or invalid tick type is expected.
BROADTAPE_SOURCES = ["BRF", "BZ", "FLY"]

WAIT_SECONDS = 600
WARMUP_SECONDS = 8
OLD_NEWS_GRACE_SECONDS = 120
FETCH_ARTICLES = True
FETCH_SNAPSHOT_ARTICLES = True
MAX_ARTICLE_REQUESTS = 50
ARTICLE_TEXT_PREVIEW = 800

print("CONTRACT_NEWS_SYMBOLS =", CONTRACT_NEWS_SYMBOLS)
print("CONTRACT_NEWS_PROVIDERS =", CONTRACT_NEWS_PROVIDERS)
print("BROADTAPE_SOURCES =", BROADTAPE_SOURCES)

# Test C: historical news. Use this when realtime has no fresh headlines.
HISTORICAL_NEWS_SYMBOLS = ["NVDA", "TSLA", "AAPL", "MSFT"]
HISTORICAL_NEWS_PROVIDERS = ["BRFG", "BRFUPDN", "DJNL"]
HISTORICAL_NEWS_COUNT = 20
HISTORICAL_END_DATETIME = ""  # empty means this notebook will fill current local time as YYYYMMDD HH:MM:SS
HISTORICAL_START_DATETIME = ""  # empty means provider default/backfill window

print("HISTORICAL_NEWS_SYMBOLS =", HISTORICAL_NEWS_SYMBOLS)
print("HISTORICAL_NEWS_PROVIDERS =", HISTORICAL_NEWS_PROVIDERS)


## 1. Raw IB client and helpers


In [ ]:
def clean_article_text(raw_text):
    if not raw_text:
        return ""
    text = html.unescape(raw_text)
    text = re.sub(r"(?is)<(script|style|noscript).*?</\1>", " ", text)
    text = re.sub(r"(?s)<br\s*/?>", "\n", text)
    text = re.sub(r"(?s)</p\s*>", "\n", text)
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def parse_ib_datetime(value):
    try:
        ts = int(value)
    except (TypeError, ValueError):
        return None
    if ts > 10_000_000_000:
        ts = ts / 1000
    return datetime.fromtimestamp(ts, timezone.utc)


def format_ib_time(value):
    dt = parse_ib_datetime(value)
    if not dt:
        return ""
    return dt.astimezone(LOCAL_TZ).isoformat()


def format_historical_time(value):
    if not value:
        return ""
    text = str(value).strip()
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y%m%d %H:%M:%S", "%Y%m%d-%H:%M:%S"):
        try:
            return datetime.strptime(text, fmt).replace(tzinfo=timezone.utc).astimezone(LOCAL_TZ).isoformat()
        except ValueError:
            pass
    return text


def stock_contract(symbol):
    contract = Contract()
    contract.symbol = symbol
    contract.secType = "STK"
    contract.exchange = "SMART"
    if symbol in {"5803", "6471"}:
        contract.currency = "JPY"
    elif symbol == "IQE":
        contract.currency = "GBP"
    elif symbol == "XFAB":
        contract.currency = "EUR"
    else:
        contract.currency = "USD"
    return contract


def broadtape_contract(source):
    source = source.upper()
    contract = Contract()
    contract.symbol = f"{source}:{source}_ALL"
    contract.secType = "NEWS"
    contract.exchange = source
    contract.currency = ""
    return contract


class NewsApiProbe(EWrapper, EClient):
    def __init__(self, name):
        EClient.__init__(self, self)
        self.name = name
        self.ready = threading.Event()
        self.news = []
        self.articles = []
        self.errors = []
        self.ticker_map = {}
        self.article_req_map = {}
        self.next_article_req_id = 9000
        self.article_request_count = 0
        self.listen_started_at = None
        self.listen_started_utc = None
        self.seen_article_keys = set()
        self.snapshot_count = 0
        self.new_count = 0
        self.duplicate_count = 0
        self.contract_details = {}
        self.contract_detail_events = {}
        self.historical_news = []
        self.historical_req_map = {}
        self.historical_events = {}

    def nextValidId(self, orderId):
        print(f"[{self.name}] IB ready, nextValidId=", orderId)
        self.ready.set()

    def newsProviders(self, providers):
        print(f"[{self.name}] visible article providers:")
        for provider in providers:
            code = getattr(provider, "providerCode", None) or getattr(provider, "code", "")
            name = getattr(provider, "providerName", None) or getattr(provider, "name", "")
            print(" ", code, "-", name)

    def classify_news(self, providerCode, articleId, timeStamp):
        key = (providerCode, articleId)
        elapsed = time.time() - self.listen_started_at if self.listen_started_at else 0
        article_dt = parse_ib_datetime(timeStamp)

        if key in self.seen_article_keys:
            return "DUPLICATE", "same provider/article_id already seen", elapsed

        if article_dt and self.listen_started_utc:
            cutoff = self.listen_started_utc - timedelta(seconds=OLD_NEWS_GRACE_SECONDS)
            if article_dt < cutoff:
                return "SNAPSHOT", "ib timestamp is older than this subscription", elapsed

        if elapsed <= WARMUP_SECONDS:
            return "SNAPSHOT", "arrived inside warmup window", elapsed

        return "NEW", "unseen article after subscription start", elapsed

    def tickNews(self, tickerId, timeStamp, providerCode, articleId, headline, extraData):
        key = (providerCode, articleId)
        phase, phase_reason, elapsed = self.classify_news(providerCode, articleId, timeStamp)
        if phase == "DUPLICATE":
            self.duplicate_count += 1
        elif phase == "SNAPSHOT":
            self.snapshot_count += 1
        else:
            self.new_count += 1
        self.seen_article_keys.add(key)

        row = {
            "mode": self.name,
            "phase": phase,
            "phase_reason": phase_reason,
            "received_at_local": datetime.now(timezone.utc).astimezone(LOCAL_TZ).isoformat(),
            "listen_started_local": self.listen_started_utc.astimezone(LOCAL_TZ).isoformat() if self.listen_started_utc else "",
            "elapsed_seconds": round(elapsed, 2),
            "ticker_id": tickerId,
            "symbol": self.ticker_map.get(tickerId, {}).get("symbol", "?"),
            "subscription": self.ticker_map.get(tickerId, {}).get("subscription", ""),
            "provider": providerCode,
            "article_id": articleId,
            "ib_timestamp": timeStamp,
            "ib_time_local": format_ib_time(timeStamp),
            "headline": headline,
            "extra_data": extraData,
        }
        self.news.append(row)
        print(f"\n[TICK_NEWS][{self.name}][{phase}]", row)

        should_fetch_article = FETCH_ARTICLES and phase != "DUPLICATE" and (phase == "NEW" or FETCH_SNAPSHOT_ARTICLES)
        if should_fetch_article and self.article_request_count < MAX_ARTICLE_REQUESTS:
            req_id = self.next_article_req_id
            self.next_article_req_id += 1
            self.article_request_count += 1
            self.article_req_map[req_id] = row
            print("[ARTICLE_REQUEST]", {"req_id": req_id, "mode": self.name, "phase": phase, "provider": providerCode, "article_id": articleId})
            self.reqNewsArticle(req_id, providerCode, articleId, [])

    def newsArticle(self, reqId, articleType, articleText):
        meta = self.article_req_map.pop(reqId, {})
        cleaned = clean_article_text(articleText)
        row = {
            "req_id": reqId,
            "article_type": articleType,
            "mode": meta.get("mode", self.name),
            "phase": meta.get("phase", ""),
            "phase_reason": meta.get("phase_reason", ""),
            "symbol": meta.get("symbol", ""),
            "provider": meta.get("provider", ""),
            "article_id": meta.get("article_id", ""),
            "ib_time_local": meta.get("ib_time_local", ""),
            "headline": meta.get("headline", ""),
            "text": cleaned,
            "raw_len": len(articleText or ""),
            "text_len": len(cleaned),
        }
        self.articles.append(row)
        preview = cleaned[:ARTICLE_TEXT_PREVIEW] if cleaned else "<empty>"
        print("\n[ARTICLE]", {k: v for k, v in row.items() if k != "text"})
        print(preview)


    def contractDetails(self, reqId, contractDetails):
        contract = contractDetails.contract
        row = {
            "req_id": reqId,
            "symbol": getattr(contract, "symbol", ""),
            "con_id": getattr(contract, "conId", 0),
            "sec_type": getattr(contract, "secType", ""),
            "exchange": getattr(contract, "exchange", ""),
            "primary_exchange": getattr(contract, "primaryExchange", ""),
            "currency": getattr(contract, "currency", ""),
            "local_symbol": getattr(contract, "localSymbol", ""),
            "long_name": getattr(contractDetails, "longName", ""),
        }
        self.contract_details.setdefault(reqId, []).append(row)
        print("[CONTRACT_DETAIL]", row)

    def contractDetailsEnd(self, reqId):
        event = self.contract_detail_events.get(reqId)
        if event:
            event.set()

    def historicalNews(self, reqId, time, providerCode, articleId, headline):
        symbol = self.historical_req_map.get(reqId, {}).get("symbol", "")
        row = {
            "req_id": reqId,
            "symbol": symbol,
            "time_raw": time,
            "time_local": format_historical_time(time),
            "provider": providerCode,
            "article_id": articleId,
            "headline": headline,
        }
        self.historical_news.append(row)
        print("[HISTORICAL_NEWS]", row)

        if FETCH_ARTICLES and self.article_request_count < MAX_ARTICLE_REQUESTS:
            article_req_id = self.next_article_req_id
            self.next_article_req_id += 1
            self.article_request_count += 1
            meta = {
                "mode": self.name,
                "phase": "HISTORICAL",
                "phase_reason": "reqHistoricalNews result",
                "symbol": symbol,
                "provider": providerCode,
                "article_id": articleId,
                "ib_time_local": row["time_local"],
                "headline": headline,
            }
            self.article_req_map[article_req_id] = meta
            print("[ARTICLE_REQUEST]", {"req_id": article_req_id, "phase": "HISTORICAL", "provider": providerCode, "article_id": articleId})
            self.reqNewsArticle(article_req_id, providerCode, articleId, [])

    def historicalNewsEnd(self, reqId, hasMore):
        print("[HISTORICAL_NEWS_END]", {"req_id": reqId, "has_more": hasMore})
        event = self.historical_events.get(reqId)
        if event:
            event.set()

    def error(self, reqId, errorTime, errorCode, errorString, advancedOrderRejectJson=""):
        msg = {
            "mode": self.name,
            "reqId": reqId,
            "code": errorCode,
            "message": errorString,
            "subscription": self.ticker_map.get(reqId, {}),
        }
        self.errors.append(msg)
        if errorCode in {2104, 2106, 2107, 2108, 2158, 2119}:
            print("[status]", msg)
        else:
            article_meta = self.article_req_map.pop(reqId, None)
            if article_meta:
                print("[article_error]", msg, article_meta)
            else:
                print("[error]", msg)


## 2. Runner and subscription functions


In [ ]:
def run_probe(name, subscribe_func, wait_seconds=WAIT_SECONDS, client_id=CLIENT_ID):
    app = NewsApiProbe(name)
    app.connect(HOST, PORT, clientId=client_id)

    thread = threading.Thread(target=app.run, daemon=True)
    thread.start()

    if not app.ready.wait(timeout=15):
        raise TimeoutError("IB handshake did not finish. Check TWS/Gateway API port and client id.")

    app.reqNewsProviders()
    time.sleep(2)

    app.listen_started_at = time.time()
    app.listen_started_utc = datetime.now(timezone.utc)
    print(f"[{name}] subscription_start_local =", app.listen_started_utc.astimezone(LOCAL_TZ).isoformat())
    print(f"[{name}] timestamp snapshot rule: article time older than start - {OLD_NEWS_GRACE_SECONDS}s is SNAPSHOT")

    subscribe_func(app)

    deadline = time.time() + wait_seconds
    try:
        while time.time() < deadline:
            print(
                f"heartbeat [{name}] "
                f"snapshot={app.snapshot_count} new={app.new_count} duplicate={app.duplicate_count} "
                f"article_count={len(app.articles)} error_count={len(app.errors)}"
            )
            time.sleep(10)
    finally:
        for ticker_id in list(app.ticker_map):
            try:
                app.cancelMktData(ticker_id)
            except Exception as exc:
                print("cancel failed", ticker_id, exc)
        app.disconnect()

    return app


def subscribe_contract_news(app, symbols=None, providers=CONTRACT_NEWS_PROVIDERS):
    symbols = symbols or CONTRACT_NEWS_SYMBOLS
    generic_ticks = f"mdoff,292:{providers}"
    for offset, symbol in enumerate(symbols):
        ticker_id = 7000 + offset
        contract = stock_contract(symbol)
        app.ticker_map[ticker_id] = {"symbol": symbol, "subscription": f"stock:{symbol}:{providers}"}
        print("subscribe CONTRACT", ticker_id, symbol, contract.secType, contract.exchange, contract.currency, generic_ticks)
        app.reqMktData(ticker_id, contract, generic_ticks, False, False, [])


def subscribe_broadtape_news(app, sources=None):
    sources = sources or BROADTAPE_SOURCES
    for offset, source in enumerate(sources):
        ticker_id = 8000 + offset
        contract = broadtape_contract(source)
        app.ticker_map[ticker_id] = {"symbol": f"ALL:{source}", "subscription": f"broadtape:{contract.symbol}"}
        print("subscribe BROADTAPE", ticker_id, contract.symbol, contract.secType, contract.exchange, "mdoff,292")
        app.reqMktData(ticker_id, contract, "mdoff,292", False, False, [])


## 3. Output helpers


In [ ]:
def print_probe_summary(probe):
    print("\nSUMMARY", probe.name)
    print("news_count =", len(probe.news))
    print("snapshot_count =", probe.snapshot_count)
    print("new_count =", probe.new_count)
    print("duplicate_count =", probe.duplicate_count)
    print("article_count =", len(probe.articles))
    print("errors =", probe.errors)


def print_headlines(probe):
    for i, item in enumerate(probe.news, 1):
        print(
            i,
            item["mode"],
            item["phase"],
            item["symbol"],
            item["provider"],
            item["article_id"],
            item["ib_time_local"],
            item["headline"],
        )


def print_articles(probe):
    for i, article in enumerate(probe.articles, 1):
        print("=" * 100)
        print(
            i,
            article["mode"],
            article["phase"],
            article["symbol"],
            article["provider"],
            article["article_id"],
            article["ib_time_local"],
            article["headline"],
        )
        print(article["text"][:1500] if article["text"] else "<empty>")


## 4. Cleanup existing clients


In [ ]:
def delete_probe(probe):
    """Disconnect a probe created in this notebook and release its IB client id.

    IB cannot kill another external process by client id. This only cleans up
    probe objects that still exist in the current notebook kernel.
    """
    if probe is None:
        print("delete_probe: probe is None")
        return

    name = getattr(probe, "name", "UNKNOWN")
    ticker_map = getattr(probe, "ticker_map", {}) or {}

    for ticker_id in list(ticker_map):
        try:
            probe.cancelMktData(ticker_id)
            print(f"[{name}] cancelled market data", ticker_id, ticker_map.get(ticker_id))
        except Exception as exc:
            print(f"[{name}] cancelMktData failed", ticker_id, exc)

    try:
        if probe.isConnected():
            probe.disconnect()
            print(f"[{name}] disconnected")
        else:
            print(f"[{name}] already disconnected")
    except Exception as exc:
        print(f"[{name}] disconnect failed", exc)


def delete_existing_probes():
    """Clean up the common probe variables used by this notebook."""
    for var_name in ("contract_probe", "broadtape_probe", "historical_probe", "probe"):
        probe = globals().get(var_name)
        if probe is not None:
            print("cleanup", var_name)
            delete_probe(probe)


## 5. Test A: contract-specific stock news


In [ ]:
# Start with a few symbols. Use symbols=None after this path is verified.
contract_probe = run_probe(
    "CONTRACT",
    lambda app: subscribe_contract_news(app, symbols=["IBM", "AVAV", "AOSL"]),
    wait_seconds=WAIT_SECONDS,
    client_id=CLIENT_ID,
)
print_probe_summary(contract_probe)


## 6. Inspect Test A


In [ ]:
print_headlines(contract_probe)
print_articles(contract_probe)


## 7. Test B: BroadTape all-news contracts


In [ ]:
# BroadTape needs separate API entitlement. code=200 usually means the NEWS contract is unavailable for this account.
broadtape_probe = run_probe(
    "BROADTAPE",
    lambda app: subscribe_broadtape_news(app, sources=BROADTAPE_SOURCES),
    wait_seconds=WAIT_SECONDS,
    client_id=CLIENT_ID + 1,
)
print_probe_summary(broadtape_probe)


## 8. Inspect Test B


In [ ]:
print_headlines(broadtape_probe)
print_articles(broadtape_probe)


## 9. Test C: historical stock news


In [ ]:
def ib_datetime_now_string():
    return datetime.now(LOCAL_TZ).strftime("%Y%m%d %H:%M:%S")


def as_provider_list(providers):
    if isinstance(providers, str):
        return [item.strip() for item in providers.replace(",", "+").split("+") if item.strip()]
    return list(providers)


def request_contract_details_sync(app, symbol, req_id, timeout=15):
    app.contract_detail_events[req_id] = threading.Event()
    app.reqContractDetails(req_id, stock_contract(symbol))
    completed = app.contract_detail_events[req_id].wait(timeout=timeout)
    rows = app.contract_details.get(req_id, [])
    app.contract_detail_events.pop(req_id, None)
    if not completed:
        print("[CONTRACT_DETAIL_TIMEOUT]", {"req_id": req_id, "symbol": symbol, "timeout": timeout, "rows": len(rows)})
    return rows


def request_historical_news_sync(app, symbol, con_id, req_id, provider, count=HISTORICAL_NEWS_COUNT, timeout=45):
    before = len(app.historical_news)
    end_time = HISTORICAL_END_DATETIME or ib_datetime_now_string()
    app.historical_req_map[req_id] = {"symbol": symbol, "con_id": con_id, "provider": provider}
    app.historical_events[req_id] = threading.Event()
    print("request HISTORICAL", {
        "req_id": req_id,
        "symbol": symbol,
        "con_id": con_id,
        "provider": provider,
        "start": HISTORICAL_START_DATETIME,
        "end": end_time,
        "count": count,
    })
    app.reqHistoricalNews(
        reqId=req_id,
        conId=con_id,
        providerCodes=provider,
        startDateTime=HISTORICAL_START_DATETIME,
        endDateTime=end_time,
        totalResults=count,
        historicalNewsOptions=[],
    )
    completed = app.historical_events[req_id].wait(timeout=timeout)
    rows = app.historical_news[before:]
    app.historical_events.pop(req_id, None)
    if not completed:
        print("[HISTORICAL_TIMEOUT]", {"req_id": req_id, "symbol": symbol, "provider": provider, "timeout": timeout, "rows": len(rows)})
    elif not rows:
        print("[HISTORICAL_EMPTY]", {"req_id": req_id, "symbol": symbol, "provider": provider})
    return rows


def run_historical_news_probe(symbols=None, providers=None, count=HISTORICAL_NEWS_COUNT, client_id=CLIENT_ID + 20):
    symbols = symbols or HISTORICAL_NEWS_SYMBOLS
    providers = as_provider_list(providers or HISTORICAL_NEWS_PROVIDERS)
    app = NewsApiProbe("HISTORICAL")
    app.connect(HOST, PORT, clientId=client_id)

    thread = threading.Thread(target=app.run, daemon=True)
    thread.start()
    if not app.ready.wait(timeout=15):
        raise TimeoutError("IB handshake did not finish. Check TWS/Gateway API port and client id.")

    app.reqNewsProviders()
    time.sleep(2)

    try:
        req_offset = 0
        for symbol_index, symbol in enumerate(symbols):
            details_req_id = 10000 + symbol_index
            details = request_contract_details_sync(app, symbol, details_req_id)
            if not details:
                print("no contract details", symbol)
                continue

            chosen = details[0]
            con_id = int(chosen["con_id"])
            print("chosen contract", symbol, chosen)

            for provider in providers:
                req_id = 11000 + req_offset
                req_offset += 1
                request_historical_news_sync(
                    app,
                    symbol,
                    con_id,
                    req_id=req_id,
                    provider=provider,
                    count=count,
                )
                time.sleep(1)

        # Give article callbacks a short window after historical headlines return.
        time.sleep(5)
    finally:
        app.disconnect()

    return app


## 10. Run Test C


In [ ]:
historical_probe = run_historical_news_probe(
    symbols=HISTORICAL_NEWS_SYMBOLS,
    providers=HISTORICAL_NEWS_PROVIDERS,
    count=HISTORICAL_NEWS_COUNT,
    client_id=CLIENT_ID + 20,
)
print("historical headlines =", len(historical_probe.historical_news))
print("articles =", len(historical_probe.articles))
print("errors =", historical_probe.errors)


## 11. Inspect Test C


In [ ]:
for i, item in enumerate(historical_probe.historical_news, 1):
    print(i, item["symbol"], item["provider"], item["article_id"], item["time_local"], item["headline"])

print_articles(historical_probe)
